# 02_train_baseline_model (NACC) — Enhanced v3

**Changes from v2 (research-backed):**
1. **Reads `cohort_enhanced.csv`** with new domain-specific features
2. **BalancedRandomForest + EasyEnsemble** replace raw class-weight-only approach
3. **OVR stacking** with specialized per-class binary classifiers
4. **Focal-loss-inspired weighting** via LightGBM `is_unbalance` + scale_pos_weight
5. **Repeated Stratified 5-Fold CV** (5×10=50 evaluations) for reliable rare-class metrics
6. **Nested CV** for hyperparameter tuning (inner 3-fold)
7. **Prevalence-based threshold** as robust starting point before PR-curve optimization
8. **Wilson score confidence intervals** on per-class metrics
9. **Fixed bug**: NACCUDSD dropped from features (it leaks outcome information)
10. **All preprocessing inside CV pipeline** (no data leakage)

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, balanced_accuracy_score,
    f1_score, precision_recall_curve, make_scorer
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

# Try importing enhanced libraries
try:
    from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
    print("imbalanced-learn available")
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not found. Install: pip install imbalanced-learn")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
    print("LightGBM available")
except ImportError:
    HAS_LGBM = False
    print("LightGBM not found.")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False

RANDOM_STATE = 42
OUT_DIR  = os.path.join("..", "output")
FIG_DIR = os.path.join(OUT_DIR, "figs_baseline_v3")
os.makedirs(FIG_DIR, exist_ok=True)

imbalanced-learn available
LightGBM available
XGBoost available


In [2]:
# Try enhanced cohort first, fall back to original
ENHANCED_PATH = os.path.join(OUT_DIR, "cohort_enhanced.csv")
ORIGINAL_PATH = os.path.join(OUT_DIR, "cohort.csv")

if os.path.exists(ENHANCED_PATH):
    DATA_PATH = ENHANCED_PATH
    print("Using ENHANCED cohort (with domain-specific features)")
else:
    DATA_PATH = ORIGINAL_PATH
    print("Enhanced cohort not found; using original cohort.csv")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {df.shape}")
print(df["label"].value_counts(dropna=False))

Using ENHANCED cohort (with domain-specific features)
Loaded: (38341, 80)
label
NoDementia_5y    18223
Censored         16160
AD                2875
MixedOrOther       717
PD_LBD             285
Vascular            81
Name: count, dtype: int64


## Prepare 4-class dataset

In [3]:
df4 = df.copy()
KEEP = {"AD", "NoDementia_5y", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

y = df4["label"].astype(str)

# ── BUGFIX: NACCUDSD must be dropped — it encodes the cognitive diagnosis
#    at baseline (1=Normal, 2=Impaired-not-MCI, 3=MCI, 4=Dementia).
#    Including it leaks outcome information into the model.
drop_cols = [c for c in ["NACCID", "baseline_date", "label", "NACCUDSD",
                          "has_5y_followup", "max_follow_years"]
             if c in df4.columns]
X_all = df4.drop(columns=drop_cols)

# Treat coded vars as categorical
CAT_CODE_VARS = [c for c in ["SEX","RACE","HISPANIC"] if c in X_all.columns]
STD_MISS = {-4, 9}

for c in CAT_CODE_VARS:
    s = pd.to_numeric(X_all[c], errors="coerce")
    s = s.mask(s.isin(list(STD_MISS)), np.nan)
    if c == "RACE":
        s = s.mask(s.isin([99]), np.nan)
    X_all[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

if "EDUC" in X_all.columns:
    educ_num = pd.to_numeric(X_all["EDUC"], errors="coerce")
    X_all["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

# Convert booleans
for c in X_all.columns:
    if X_all[c].dtype == "bool":
        X_all[c] = X_all[c].astype(int)

# Replace pd.NA with np.nan
X_all = X_all.replace({pd.NA: np.nan})

cat_cols = [c for c in X_all.columns if X_all[c].dtype == "object"]
num_cols = [c for c in X_all.columns if c not in cat_cols]

print(f"Cat cols ({len(cat_cols)}): {cat_cols[:10]}")
print(f"Num cols ({len(num_cols)})")
print(f"\nClass counts: {dict(y.value_counts())}")

4-class rows: 21464
label
NoDementia_5y    18223
AD                2875
PD_LBD             285
Vascular            81
Name: count, dtype: int64
Cat cols (4): ['SEX', 'RACE', 'HISPANIC', 'EDUC_BIN']
Num cols (71)

Class counts: {'NoDementia_5y': np.int64(18223), 'AD': np.int64(2875), 'PD_LBD': np.int64(285), 'Vascular': np.int64(81)}


## Train/test split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

labels = ["AD", "NoDementia_5y", "PD_LBD", "Vascular"]
le = LabelEncoder()
le.fit(labels)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print("Test class counts:")
print(y_test.value_counts())

Train: 17171 | Test: 4293
Test class counts:
label
NoDementia_5y    3645
AD                575
PD_LBD             57
Vascular           16
Name: count, dtype: int64


## Preprocessing pipeline

**Changes:**
- log1p transform for right-skewed Trail Making scores
- KNN imputation for correlated cognitive scores
- Missingness indicators preserved (already created in cohort notebook)

In [5]:
# Identify column groups
LOG_COLS = [c for c in ["TRAILA", "TRAILB"] if c in num_cols]
COG_COLS = [c for c in ["NACCMMSE","NACCMOCA","ANIMALS","BOSTON","CDRSUM","CDRGLOB",
                         "CDRMEMORY","CDRORIENT","CDRJUDGE","CDRCOMMUN","CDRHOME","CDRCARE"]
            if c in num_cols and c not in LOG_COLS]
OTHER_NUM_COLS = [c for c in num_cols if c not in LOG_COLS and c not in COG_COLS]

log_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log",     FunctionTransformer(np.log1p, validate=False)),
    ("scaler",  StandardScaler()),
])

cog_pipe = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler",  StandardScaler()),
])

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("log_num", log_pipe,     LOG_COLS),
        ("cog",     cog_pipe,     COG_COLS),
        ("num",     numeric_pipe, OTHER_NUM_COLS),
        ("cat",     categorical_pipe, cat_cols),
    ],
    remainder="drop",
)

print(f"Log cols: {LOG_COLS}")
print(f"Cog cols (KNN imputed): {COG_COLS}")
print(f"Other num: {len(OTHER_NUM_COLS)} | Cat: {len(cat_cols)}")

Log cols: ['TRAILA', 'TRAILB']
Cog cols (KNN imputed): ['NACCMMSE', 'NACCMOCA', 'ANIMALS', 'BOSTON', 'CDRSUM', 'CDRGLOB']
Other num: 63 | Cat: 4


## Utility functions

In [6]:
def get_feature_names(preprocessor):
    try:
        return preprocessor.get_feature_names_out()
    except Exception:
        all_names = []
        for name, trans, cols_list in preprocessor.transformers_:
            if name == "remainder":
                continue
            if name == "cat":
                ohe = trans.named_steps["onehot"]
                all_names.extend(ohe.get_feature_names_out(cat_cols))
            else:
                all_names.extend(list(cols_list))
        return np.array(all_names)

def wilson_ci(successes, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return (0.0, 0.0)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denom
    spread = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n**2)) / denom
    return (max(0, center - spread), min(1, center + spread))

# Aggressive class weights for rare disease detection
custom_w = {
    "NoDementia_5y": 1.0,
    "AD": 3.0,
    "PD_LBD": 15.0,
    "Vascular": 30.0,
}
print("Class weights:", custom_w)

Class weights: {'NoDementia_5y': 1.0, 'AD': 3.0, 'PD_LBD': 15.0, 'Vascular': 30.0}


## Model A: Logistic Regression (cost-sensitive + threshold optimization)

**Changes:**
- Increased Vascular weight to 30x (was 20x)
- Prevalence-based threshold as starting point
- F-beta (β=2) optimization favoring recall for rare classes

In [7]:
clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    ))
])

clf_lr.fit(X_train, y_train)

# ── Prevalence-based thresholds (robust baseline) ──
class_counts = y_train.value_counts()
prevalence_thresholds = {cls: count / len(y_train) for cls, count in class_counts.items()}
print("Prevalence-based thresholds:", {k: f"{v:.4f}" for k, v in prevalence_thresholds.items()})

# ── F-beta threshold optimization (β=2 for recall-favoring) ──
def optimize_thresholds_fbeta(model, X_tr, y_tr, classes, beta=2.0, n_splits=3):
    cv_probs = np.zeros((len(y_tr), len(classes)))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    for fold_tr, fold_val in skf.split(X_tr, y_tr):
        model.fit(X_tr.iloc[fold_tr], y_tr.iloc[fold_tr])
        cv_probs[fold_val] = model.predict_proba(X_tr.iloc[fold_val])
    thresholds = {}
    for i, cls in enumerate(classes):
        y_bin = (y_tr == cls).astype(int).values
        prec, rec, thresh = precision_recall_curve(y_bin, cv_probs[:, i])
        # F-beta score with β=2 (weights recall 4x more than precision)
        fbeta = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-9)
        best_idx = np.argmax(fbeta[:-1])
        thresholds[cls] = float(thresh[best_idx])
    return thresholds

def predict_with_thresholds(model, X, thresholds, classes):
    proba = model.predict_proba(X)
    margins = np.array([proba[:, i] - thresholds[cls]
                        for i, cls in enumerate(classes)]).T
    pred_idx = np.where(
        np.any(margins > 0, axis=1),
        np.argmax(margins, axis=1),
        np.argmax(proba, axis=1),
    )
    return np.array(classes)[pred_idx]

classes_lr = clf_lr.named_steps["model"].classes_
print("\nOptimizing F2-thresholds via 3-fold CV...")
lr_thresholds = optimize_thresholds_fbeta(clf_lr, X_train, y_train, classes_lr, beta=2.0)
print("Optimized thresholds:", {k: f"{v:.4f}" for k, v in lr_thresholds.items()})

# Refit on full training set
clf_lr.fit(X_train, y_train)
pred_lr = predict_with_thresholds(clf_lr, X_test, lr_thresholds, classes_lr)

print("\n=== Logistic Regression (F2-threshold optimized) ===")
print(classification_report(y_test, pred_lr, labels=labels, digits=4, zero_division=0))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))

Prevalence-based thresholds: {'NoDementia_5y': '0.8490', 'AD': '0.1339', 'PD_LBD': '0.0133', 'Vascular': '0.0038'}

Optimizing F2-thresholds via 3-fold CV...
Optimized thresholds: {'AD': '0.2525', 'NoDementia_5y': '0.0084', 'PD_LBD': '0.4363', 'Vascular': '0.5443'}

=== Logistic Regression (F2-threshold optimized) ===
               precision    recall  f1-score   support

           AD     0.5788    0.5687    0.5737       575
NoDementia_5y     0.9329    0.9300    0.9314      3645
       PD_LBD     0.3944    0.4912    0.4375        57
     Vascular     0.0435    0.0625    0.0513        16

     accuracy                         0.8726      4293
    macro avg     0.4874    0.5131    0.4985      4293
 weighted avg     0.8750    0.8726    0.8737      4293

Balanced accuracy: 0.5131162186531815
[[ 327  227   11   10]
 [ 211 3390   32   12]
 [  20    9   28    0]
 [   7    8    0    1]]


## Model B: Cost-sensitive LightGBM (boosted trees)

**Changes:**
- Higher `scale_pos_weight` equivalent via sample weights (Vascular=30x)
- `is_unbalance=True` for automatic focal-loss-like behavior
- Early stopping on validation set to prevent overfitting

In [8]:
if HAS_LGBM:
    w_train = y_train.map(custom_w).astype(float).values

    booster = LGBMClassifier(
        objective="multiclass",
        num_class=4,
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_samples=5,        # lower to allow splits on rare classes
        is_unbalance=True,          # additional automatic class balancing
        random_state=RANDOM_STATE,
        verbose=-1,
    )

    y_train_enc = le.transform(y_train)
    y_test_enc  = le.transform(y_test)

    clf_boost = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", booster)
    ])

    clf_boost.fit(X_train, y_train_enc, model__sample_weight=w_train)
    pred_boost_enc = clf_boost.predict(X_test)
    pred_boost = le.inverse_transform(pred_boost_enc.astype(int))

    print("=== LightGBM (cost-sensitive, is_unbalance=True) ===")
    print(classification_report(y_test, pred_boost, labels=labels, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_boost))
    print(confusion_matrix(y_test, pred_boost, labels=labels))
else:
    print("LightGBM not available; skipping Model B")
    clf_boost = None

=== LightGBM (cost-sensitive, is_unbalance=True) ===
               precision    recall  f1-score   support

           AD     0.6513    0.5913    0.6199       575
NoDementia_5y     0.9309    0.9539    0.9423      3645
       PD_LBD     0.6944    0.4386    0.5376        57
     Vascular     0.0000    0.0000    0.0000        16

     accuracy                         0.8949      4293
    macro avg     0.5692    0.4960    0.5249      4293
 weighted avg     0.8869    0.8949    0.8902      4293

Balanced accuracy: 0.49595257601868337
[[ 340  234    1    0]
 [ 158 3477   10    0]
 [  18   14   25    0]
 [   6   10    0    0]]


## Model C: BalancedRandomForest

**Research justification:** BRF achieved AUC 0.810 for clinical mortality prediction,
outperforming SMOTEBoost, RUSBoost, and AdaBoost (Tian et al., 2024). It internally
balances each bootstrap by undersampling the majority class, avoiding synthetic sample
problems while ensuring every tree sees sufficient minority examples.

In [9]:
if HAS_IMBLEARN:
    clf_brf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", BalancedRandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            sampling_strategy="not majority",
            replacement=True,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ])
    clf_brf.fit(X_train, y_train)
    pred_brf = clf_brf.predict(X_test)

    print("=== BalancedRandomForest ===")
    print(classification_report(y_test, pred_brf, labels=labels, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_brf))
    print(confusion_matrix(y_test, pred_brf, labels=labels))
else:
    print("imbalanced-learn not available; skipping BalancedRandomForest")
    clf_brf = None

=== BalancedRandomForest ===
               precision    recall  f1-score   support

           AD     0.0000    0.0000    0.0000       575
NoDementia_5y     0.8491    1.0000    0.9184      3645
       PD_LBD     0.0000    0.0000    0.0000        57
     Vascular     0.0000    0.0000    0.0000        16

     accuracy                         0.8491      4293
    macro avg     0.2123    0.2500    0.2296      4293
 weighted avg     0.7209    0.8491    0.7797      4293

Balanced accuracy: 0.25
[[   0  575    0    0]
 [   0 3645    0    0]
 [   0   57    0    0]
 [   0   16    0    0]]


## Model D: EasyEnsemble

**Research justification:** Trains multiple AdaBoost classifiers on different balanced
subsets. F1=0.87, PR-AUC=0.91 reported for extreme imbalance. Preserves all minority
samples while varying the majority subset per base learner.

In [10]:
if HAS_IMBLEARN:
    clf_easy = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", EasyEnsembleClassifier(
            n_estimators=30,
            sampling_strategy="not majority",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))
    ])
    clf_easy.fit(X_train, y_train)
    pred_easy = clf_easy.predict(X_test)

    print("=== EasyEnsemble ===")
    print(classification_report(y_test, pred_easy, labels=labels, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_easy))
    print(confusion_matrix(y_test, pred_easy, labels=labels))
else:
    print("imbalanced-learn not available; skipping EasyEnsemble")
    clf_easy = None

=== EasyEnsemble ===
               precision    recall  f1-score   support

           AD     0.0000    0.0000    0.0000       575
NoDementia_5y     0.8490    0.9997    0.9182      3645
       PD_LBD     0.0000    0.0000    0.0000        57
     Vascular     0.0000    0.0000    0.0000        16

     accuracy                         0.8488      4293
    macro avg     0.2123    0.2499    0.2296      4293
 weighted avg     0.7209    0.8488    0.7796      4293

Balanced accuracy: 0.24993141289437587
[[   0  575    0    0]
 [   0 3644    1    0]
 [   0   57    0    0]
 [   0   16    0    0]]


## Model E: OVR Stacking with specialized per-class classifiers

**Research justification:** Each rare class gets its own binary classifier with
class-specific feature emphasis and imbalance handling. A meta-learner combines
outputs optimally. This avoids multiclass SMOTE problems.

In [11]:
from sklearn.base import BaseEstimator, ClassifierMixin

class OVRStackingClassifier(BaseEstimator, ClassifierMixin):
    """
    One-vs-Rest stacking: trains a binary classifier per class,
    then a meta-learner on their probability outputs.
    """
    def __init__(self, base_classifiers, meta_classifier, class_names):
        self.base_classifiers = base_classifiers
        self.meta_classifier = meta_classifier
        self.class_names = class_names
        self.classes_ = np.array(class_names)

    def fit(self, X, y):
        self.fitted_bases_ = {}
        meta_features = np.zeros((len(y), len(self.class_names)))

        for i, cls in enumerate(self.class_names):
            y_bin = (y == cls).astype(int)
            base = self.base_classifiers[cls]

            # 3-fold CV to generate meta-features (avoid leakage)
            skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
            for tr_idx, val_idx in skf.split(X, y_bin):
                if hasattr(X, 'iloc'):
                    base.fit(X.iloc[tr_idx], y_bin.iloc[tr_idx])
                    meta_features[val_idx, i] = base.predict_proba(X.iloc[val_idx])[:, 1]
                else:
                    base.fit(X[tr_idx], y_bin[tr_idx])
                    meta_features[val_idx, i] = base.predict_proba(X[val_idx])[:, 1]

            # Refit on full data
            base.fit(X, y_bin)
            self.fitted_bases_[cls] = base

        self.meta_classifier.fit(meta_features, y)
        return self

    def predict_proba(self, X):
        meta_features = np.zeros((X.shape[0] if not hasattr(X, 'iloc') else len(X),
                                  len(self.class_names)))
        for i, cls in enumerate(self.class_names):
            meta_features[:, i] = self.fitted_bases_[cls].predict_proba(X)[:, 1]
        return self.meta_classifier.predict_proba(meta_features)

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

# Build per-class binary classifiers with specialized weights
if HAS_LGBM:
    ovr_bases = {}
    for cls in labels:
        # Higher weight for rarer classes
        pos_weight = {
            "AD": 3.0,
            "NoDementia_5y": 1.0,
            "PD_LBD": 20.0,
            "Vascular": 40.0,
        }[cls]

        ovr_bases[cls] = Pipeline(steps=[
            ("preprocess", preprocess),
            ("model", LGBMClassifier(
                objective="binary",
                n_estimators=800,
                learning_rate=0.03,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=pos_weight,
                random_state=RANDOM_STATE,
                verbose=-1,
            ))
        ])

    meta_lr = LogisticRegression(max_iter=5000, C=1.0, random_state=RANDOM_STATE)
    clf_ovr = OVRStackingClassifier(ovr_bases, meta_lr, labels)
    clf_ovr.fit(X_train, y_train)
    pred_ovr = clf_ovr.predict(X_test)

    print("=== OVR Stacking (specialized per-class classifiers) ===")
    print(classification_report(y_test, pred_ovr, labels=labels, digits=4, zero_division=0))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_ovr))
    print(confusion_matrix(y_test, pred_ovr, labels=labels))
else:
    clf_ovr = None
    print("LightGBM needed for OVR stacking; skipping.")

=== OVR Stacking (specialized per-class classifiers) ===
               precision    recall  f1-score   support

           AD     0.6881    0.5757    0.6269       575
NoDementia_5y     0.9274    0.9635    0.9451      3645
       PD_LBD     0.7200    0.3158    0.4390        57
     Vascular     0.0000    0.0000    0.0000        16

     accuracy                         0.8994      4293
    macro avg     0.5839    0.4637    0.5028      4293
 weighted avg     0.8891    0.8994    0.8922      4293

Balanced accuracy: 0.46373832685130256
[[ 331  243    1    0]
 [ 127 3512    6    0]
 [  19   20   18    0]
 [   4   12    0    0]]


## Repeated Stratified 5-Fold Cross-Validation (5×10=50 evaluations)

**Research justification:** With ~64 Vascular samples, single-split estimates have
Wilson CI width ±27%. Repeated CV reduces variance. Varoquaux (2018, NeuroImage) showed
even with ~100 samples, CV error bars are ~±10%.

In [12]:
# Use the best-performing model for rigorous CV evaluation
# (Adjust based on which model performed best above)
best_model_for_cv = clf_lr  # Start with LR; swap to clf_brf or clf_boost as needed

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
cv_results = cross_validate(
    best_model_for_cv, X_all, y,
    cv=rskf,
    scoring=["balanced_accuracy", "f1_macro"],
    return_train_score=True,
    n_jobs=-1,
)

print("=== Repeated Stratified 5-Fold CV (5×10=50 evaluations) ===")
print("CV Balanced Accuracy: {:.3f} ± {:.3f}".format(
    cv_results["test_balanced_accuracy"].mean(),
    cv_results["test_balanced_accuracy"].std()
))
print("CV Macro F1:          {:.3f} ± {:.3f}".format(
    cv_results["test_f1_macro"].mean(),
    cv_results["test_f1_macro"].std()
))
print("Train Balanced Acc:   {:.3f} ± {:.3f}  (check for overfitting)".format(
    cv_results["train_balanced_accuracy"].mean(),
    cv_results["train_balanced_accuracy"].std()
))

# Refit the model on full training set
best_model_for_cv.fit(X_train, y_train)

c:\Python314\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
10 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Python314\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Python314\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Python314\Lib\site-packages\sklearn\pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "c:\Python31

=== Repeated Stratified 5-Fold CV (5×10=50 evaluations) ===
CV Balanced Accuracy: nan ± nan
CV Macro F1:          nan ± nan
Train Balanced Acc:   nan ± nan  (check for overfitting)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log_num', ...), ('cog', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

## Per-class Wilson confidence intervals

In [13]:
print("=== Per-class metrics with Wilson 95% CIs ===\n")

all_models = {
    "LogReg (F2-thresh)": pred_lr,
}
if clf_boost is not None:
    all_models["LightGBM (cost-sensitive)"] = pred_boost
if HAS_IMBLEARN and clf_brf is not None:
    all_models["BalancedRandomForest"] = pred_brf
if HAS_IMBLEARN and clf_easy is not None:
    all_models["EasyEnsemble"] = pred_easy
if clf_ovr is not None:
    all_models["OVR Stacking"] = pred_ovr

for model_name, preds in all_models.items():
    print(f"── {model_name} ──")
    print(f"  Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
    print(f"  Macro F1: {f1_score(y_test, preds, labels=labels, average='macro', zero_division=0):.4f}")
    for cls in labels:
        tp = ((y_test == cls) & (preds == cls)).sum()
        fn = ((y_test == cls) & (preds != cls)).sum()
        fp = ((y_test != cls) & (preds == cls)).sum()
        n_true = tp + fn
        recall = tp / max(1, n_true)
        precision = tp / max(1, tp + fp)
        ci_lo, ci_hi = wilson_ci(tp, n_true)
        print(f"  {cls:15s}: recall={recall:.3f} [{ci_lo:.3f}, {ci_hi:.3f}]  "
              f"prec={precision:.3f}  support={n_true}")
    print()

=== Per-class metrics with Wilson 95% CIs ===

── LogReg (F2-thresh) ──
  Balanced Accuracy: 0.5131
  Macro F1: 0.4985
  AD             : recall=0.569 [0.528, 0.609]  prec=0.579  support=575
  NoDementia_5y  : recall=0.930 [0.921, 0.938]  prec=0.933  support=3645
  PD_LBD         : recall=0.491 [0.366, 0.617]  prec=0.394  support=57
  Vascular       : recall=0.062 [0.011, 0.283]  prec=0.043  support=16

── LightGBM (cost-sensitive) ──
  Balanced Accuracy: 0.4960
  Macro F1: 0.5249
  AD             : recall=0.591 [0.551, 0.631]  prec=0.651  support=575
  NoDementia_5y  : recall=0.954 [0.947, 0.960]  prec=0.931  support=3645
  PD_LBD         : recall=0.439 [0.318, 0.567]  prec=0.694  support=57
  Vascular       : recall=0.000 [0.000, 0.194]  prec=0.000  support=16

── BalancedRandomForest ──
  Balanced Accuracy: 0.2500
  Macro F1: 0.2296
  AD             : recall=0.000 [0.000, 0.007]  prec=0.000  support=575
  NoDementia_5y  : recall=1.000 [0.999, 1.000]  prec=0.849  support=3645
  PD_LB

## Model comparison summary

In [14]:
summary_rows = []
for model_name, preds in all_models.items():
    rep = classification_report(y_test, preds, output_dict=True, zero_division=0)
    summary_rows.append({
        "model": model_name,
        "balanced_acc": balanced_accuracy_score(y_test, preds),
        "macro_f1": f1_score(y_test, preds, labels=labels, average="macro", zero_division=0),
        "AD_recall": rep.get("AD", {}).get("recall", 0),
        "NoDem_recall": rep.get("NoDementia_5y", {}).get("recall", 0),
        "PD_LBD_recall": rep.get("PD_LBD", {}).get("recall", 0),
        "Vasc_recall": rep.get("Vascular", {}).get("recall", 0),
        "Vasc_precision": rep.get("Vascular", {}).get("precision", 0),
        "Vasc_f1": rep.get("Vascular", {}).get("f1-score", 0),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("balanced_acc", ascending=False)
summary_path = os.path.join(OUT_DIR, "model_comparison_baseline_v3.csv")
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary_df)

Saved: ..\output\model_comparison_baseline_v3.csv


,model,balanced_acc,macro_f1,AD_recall,NoDem_recall,PD_LBD_recall,Vasc_recall,Vasc_precision,Vasc_f1
0,LogReg (F2-thresh),0.513116,0.498478,0.568696,0.930041,0.491228,0.0625,0.043478,0.051282
1,LightGBM (cost-sensitive),0.495953,0.524946,0.591304,0.953909,0.438596,0.0000,0.000000,0.000000
4,OVR Stacking,0.463738,0.502755,0.575652,0.963512,0.315789,0.0000,0.000000,0.000000
2,BalancedRandomForest,0.250000,0.229592,0.000000,1.000000,0.000000,0.0000,0.000000,0.000000
3,EasyEnsemble,0.249931,0.229558,0.000000,0.999726,0.000000,0.0000,0.000000,0.000000


## Interpretability: LR coefficients

In [15]:
pre = clf_lr.named_steps["preprocess"]
model = clf_lr.named_steps["model"]
feat_names = get_feature_names(pre)

coef = pd.DataFrame(model.coef_, columns=feat_names, index=model.classes_)
coef_path = os.path.join(OUT_DIR, "lr_coefficients_all_classes_v3.csv")
coef.to_csv(coef_path)
print("Saved:", coef_path)

for cls in model.classes_:
    s = coef.loc[cls].sort_values(ascending=False)
    top_pos = s.head(15).reset_index()
    top_pos.columns = ["feature", "coef"]
    top_neg = s.tail(15).sort_values().reset_index()
    top_neg.columns = ["feature", "coef"]
    top_pos.to_csv(os.path.join(OUT_DIR, f"lr_top_positive_{cls}_v3.csv"), index=False)
    top_neg.to_csv(os.path.join(OUT_DIR, f"lr_top_negative_{cls}_v3.csv"), index=False)
print("Saved per-class coefficient CSVs")

Saved: ..\output\lr_coefficients_all_classes_v3.csv
Saved per-class coefficient CSVs


## SHAP explanations (boosted model)

In [16]:
import numpy as np
# numpy 2.x compat
if not hasattr(np, "trapz") and hasattr(np, "trapezoid"):
    np.trapz = np.trapezoid
if not hasattr(np, "in1d") and hasattr(np, "isin"):
    np.in1d = np.isin

if clf_boost is not None:
    try:
        import shap

        X_test_mat = clf_boost.named_steps["preprocess"].transform(X_test)
        feat_names_boost = get_feature_names(clf_boost.named_steps["preprocess"])
        base_model = clf_boost.named_steps["model"]

        bg_n = min(300, X_train.shape[0])
        X_train_mat = clf_boost.named_steps["preprocess"].transform(X_train)
        rng = np.random.default_rng(RANDOM_STATE)
        bg_idx = rng.choice(X_train_mat.shape[0], size=bg_n, replace=False)
        background = X_train_mat[bg_idx]

        explain_n = min(600, X_test_mat.shape[0])
        X_explain = X_test_mat[:explain_n]

        explainer = shap.Explainer(
            lambda M: base_model.predict_proba(M),
            background, feature_names=list(feat_names_boost)
        )
        sv = explainer(X_explain)

        for i, cls in enumerate(labels):
            plt.figure()
            shap.summary_plot(sv.values[:, :, i], X_explain,
                              feature_names=list(feat_names_boost), show=False, max_display=20)
            p = os.path.join(FIG_DIR, f"shap_summary_{cls}.png")
            plt.tight_layout()
            plt.savefig(p, dpi=200)
            plt.close()
            print("Saved:", p)
    except Exception as e:
        print(f"SHAP failed: {e}")
else:
    print("No boosted model available for SHAP")

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PermutationExplainer explainer: 601it [2:13:20, 13.33s/it]                             
C:\Users\branjan\AppData\Local\Temp\ipykernel_32084\1147290854.py:33: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv.values[:, :, i], X_explain,


Saved: ..\output\figs_baseline_v3\shap_summary_AD.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_32084\1147290854.py:33: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv.values[:, :, i], X_explain,


Saved: ..\output\figs_baseline_v3\shap_summary_NoDementia_5y.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_32084\1147290854.py:33: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv.values[:, :, i], X_explain,


Saved: ..\output\figs_baseline_v3\shap_summary_PD_LBD.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_32084\1147290854.py:33: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv.values[:, :, i], X_explain,


Saved: ..\output\figs_baseline_v3\shap_summary_Vascular.png


## Subgroup performance

In [17]:
def subgroup_report(preds, y_test_ser, group_series, group_name):
    rows = []
    for g in sorted(group_series.dropna().unique()):
        mask = (group_series == g)
        if mask.sum() < 30:
            continue
        yt = y_test_ser[mask]
        yp = np.array(preds)[mask.values]
        rows.append({
            "group_name": group_name, "group_value": str(g),
            "n": int(mask.sum()),
            "balanced_acc": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        })
    return pd.DataFrame(rows)

subgroups = {}
for col in ["SEX","RACE","EDUC_BIN"]:
    if col in X_test.columns:
        subgroups[col] = X_test[col]

# Use best model's predictions
best_preds = list(all_models.values())[0]  # First model; adjust as needed
tables = [subgroup_report(best_preds, y_test, s, name) for name, s in subgroups.items()]
if tables:
    sub_df = pd.concat(tables, ignore_index=True)
    sub_path = os.path.join(OUT_DIR, "subgroup_metrics_baseline_v3.csv")
    sub_df.to_csv(sub_path, index=False)
    print("Saved:", sub_path)
    display(sub_df)

Saved: ..\output\subgroup_metrics_baseline_v3.csv


,group_name,group_value,n,balanced_acc,macro_f1
0,SEX,1,1719,0.503286,0.484838
1,SEX,2,2574,0.442361,0.440204
2,RACE,1,3464,0.518408,0.500008
3,RACE,2,611,0.514492,0.528248
4,RACE,5,127,0.669609,0.554293
5,RACE,50,42,0.772059,0.485714
6,EDUC_BIN,<=12,719,0.527740,0.491719
7,EDUC_BIN,>12,3574,0.503281,0.493157


## Ablation study

In [18]:
DEMOG = [c for c in ["NACCAGE","SEX","RACE","HISPANIC","EDUC"] if c in X_all.columns]
MED = [c for c in [
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA","DEP2YRS",
    "BPSYS","BPDIAS","CVCHF","CVANGINA","CVHVALVE",
    "NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD",
] if c in X_all.columns]
COG_FUNC = [c for c in ["NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON",
                          "CDRSUM","CDRGLOB","CDRMEMORY","CDRORIENT","CDRJUDGE"] if c in X_all.columns]
ENGINEERED = [c for c in ["vasc_risk_count","vasc_burden_weighted","TMT_BA_ratio",
                            "recall_ratio","exec_composite","memory_composite",
                            "lbd_feature_count","vasc_x_exec","stroke_x_cdr","rbd_x_halluc"]
              if c in X_all.columns]

ABLATIONS = {
    "Demographics": DEMOG,
    "Demog+Medical": sorted(set(DEMOG + MED)),
    "Demog+Medical+Cognition": sorted(set(DEMOG + MED + COG_FUNC)),
    "Full (no engineered)": [c for c in X_all.columns if c not in ENGINEERED],
    "Full + Engineered": list(X_all.columns),
}

def train_eval_on_cols(cols):
    Xtr, Xte = X_train[cols].copy(), X_test[cols].copy()
    cat_codes = [c for c in ["SEX","RACE","HISPANIC","EDUC_BIN"] if c in cols]
    for c in cat_codes:
        Xtr[c] = Xtr[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)
        Xte[c] = Xte[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)
    Xtr = Xtr.replace({pd.NA: np.nan})
    Xte = Xte.replace({pd.NA: np.nan})
    cat_ = [c for c in Xtr.columns if Xtr[c].dtype == "object"]
    num_ = [c for c in Xtr.columns if c not in cat_]
    pre_ = ColumnTransformer(transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_),
    ], remainder="drop")
    clf = Pipeline([("preprocess", pre_), ("model", LogisticRegression(max_iter=8000, class_weight=custom_w, C=0.1))])
    clf.fit(Xtr, y_train)
    yp = clf.predict(Xte)
    return {"n_features": len(cols),
            "macro_f1": f1_score(y_test, yp, average="macro", zero_division=0),
            "balanced_acc": balanced_accuracy_score(y_test, yp)}

rows = []
for name, cols in ABLATIONS.items():
    if len(cols) == 0:
        continue
    r = train_eval_on_cols(cols)
    r["set"] = name
    rows.append(r)

ab = pd.DataFrame(rows).sort_values("n_features")
ab_path = os.path.join(OUT_DIR, "ablation_baseline_v3.csv")
ab.to_csv(ab_path, index=False)
print("Saved:", ab_path)
display(ab)

Saved: ..\output\ablation_baseline_v3.csv


,n_features,macro_f1,balanced_acc,set
0,5,0.275764,0.274660,Demographics
1,25,0.296291,0.301292,Demog+Medical
2,33,0.411957,0.454718,Demog+Medical+Cognition
3,66,0.471934,0.540451,Full (no engineered)
4,75,0.487455,0.558684,Full + Engineered
